## A particle inside a simplified tokamak

We evolve a single charged particle under $m\dot{\mathbf v}=Q[\mathbf E(\mathbf x,t)+\mathbf v\times\mathbf B(\mathbf x)]$, with $\mathbf E=0$ by default. Units are normalized. The field is prescribed; this is a test-particle model, without plasma collisions, turbulence, or a self-consistent plasma equilibrium.

In cylindrical coordinates $(R,\phi,Z)$, let $R_0$ be the major radius and $a$ the circular vessel's minor radius. Use the axisymmetric field
$$
B_\phi=\frac{B_0R_0}{R},\qquad
B_R=-\frac{B_0Z}{q_sR},\qquad
B_Z=\frac{B_0(R-R_0)}{q_sR}.
$$
The dominant toroidal field circles the central hole; the weaker poloidal field twists around the tube. This field is divergence-free and tangent to circular surfaces $(R-R_0)^2+Z^2=\mathrm{constant}$. Here $q_s$ controls the twist and approximates the safety factor near the magnetic axis; it is distinct from particle charge $Q$.

The particle follows its full Lorentz orbit, including gyromotion and magnetic drifts; it is **not constrained to a field line**. A position-dependent Boris update uses the field at the midpoint of a half-step drift. The simulation stops when the particle first reaches the vessel wall, resolved to a time step. The default orbit stays inside for the simulated interval; this does not imply indefinite confinement.

Background: [Tokamak magnetic geometry — Richard Fitzpatrick, University of Texas](https://farside.ph.utexas.edu/teaching/plasma1/Fusionhtml/node13.html).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.ticker import MaxNLocator
from IPython.display import HTML, Video, display
from pathlib import Path


## Motion of particle in tokamak

### 1. Full equation of motion and energy

For charge $Q$ and mass $m$, the nonrelativistic equations are
$$
\dot{\mathbf{x}}=\mathbf{v},\qquad
m\dot{\mathbf{v}}=Q\left[\mathbf{E}(\mathbf{x},t)+\mathbf{v}\times\mathbf{B}(\mathbf{x},t)\right].
$$
The kinetic energy $K=\tfrac12m|\mathbf{v}|^2$ satisfies
$$
\frac{dK}{dt}=Q\,\mathbf{E}\cdot\mathbf{v},
$$
because $\mathbf{v}\cdot(\mathbf{v}\times\mathbf{B})=0$. A magnetic field bends the orbit but cannot change its kinetic energy. An electric field accelerates or decelerates the particle according to the sign of $Q\mathbf{E}\cdot\mathbf{v}$.

**Current implementation:** the code below evaluates a parametrized electric field and includes its electric half-kicks in the Boris update. `E_params` initializes `E_phi0`, `k_E`, `omega`, and `phase`; zero amplitudes recover $\mathbf E=0$. All quantities below use the notebook's normalized units.

### 2. What does “wobble” mean?

Separate rapid **gyromotion** around the local field from slower motion of the **guiding centre**. Define
$$
B=|\mathbf{B}|,\qquad \mathbf{b}=\mathbf{B}/B,\qquad
v_\parallel=\mathbf{v}\cdot\mathbf{b},\qquad
\mathbf{v}_\perp=\mathbf{v}-v_\parallel\mathbf{b}.
$$
For a locally uniform magnetic field with negligible electric drift,
$$
\Omega_c=\frac{|Q|B}{m},\qquad
T_c=\frac{2\pi}{\Omega_c},\qquad
\rho_L=\frac{m|\mathbf{v}_\perp|}{|Q|B},\qquad
\alpha=\operatorname{atan2}(|\mathbf{v}_\perp|,v_\parallel).
$$
Here $\rho_L$ is the gyro-orbit radius, $\Omega_c$ its angular frequency, and $\alpha$ the pitch angle. The sign of $Q$ sets the rotation direction. Increasing $B$ at fixed perpendicular speed makes the wobble smaller and faster. With appreciable drift, calculate the gyro speed relative to the moving guiding centre rather than interpreting all perpendicular motion as gyration.

A useful local estimate, for $Q\ne0$, is
$$
\mathbf{X}_{\rm gc}\approx\mathbf{x}
 +\frac{m}{QB}\,\mathbf{v}_\perp\times\mathbf{b}.
$$
It is exact for uniform $\mathbf{B}$ and $\mathbf{E}=0$, and approximate in a tokamak. Averaging the orbit over gyroperiods also helps distinguish rapid wobble from slow orbit displacement.

### 3. Parallel acceleration and magnetic trapping

Write $E_\parallel=\mathbf{E}\cdot\mathbf{b}$. In the slowly varying, magnetized limit, the leading parallel equation is
$$
m\frac{dv_\parallel}{dt}\approx QE_\parallel-\mu\,\mathbf{b}\cdot\nabla B,
\qquad
\mu=\frac{m v_{\perp,\rm gyro}^2}{2B}.
$$
The first term is electric acceleration. The second is the magnetic mirror force: motion into stronger $B$ can convert parallel energy into perpendicular gyro-energy and reverse $v_\parallel$. The magnetic moment $\mu$ is approximately conserved when the field changes little over a gyro-orbit and gyroperiod. It is not an exact invariant of an arbitrary orbit or driven field.

For a static field, $\mathbf{E}=0$, and conserved $K$ and $\mu$, reflection is possible if
$$
\sin^2\alpha_0\geq \frac{B_{\rm initial}}{B_{\rm max}},
$$
where $B_{\rm max}$ is the largest field encountered along the accessible field line. Passing particles circulate toroidally; trapped particles bounce between mirror points and can produce banana-shaped guiding-centre orbits. Whether a particular initial condition is trapped must be checked from its trajectory.

### 4. Electric and magnetic drifts

The leading drift caused by a perpendicular electric field is
$$
\mathbf{v}_E=\frac{\mathbf{E}\times\mathbf{B}}{B^2}.
$$
It has the same direction for positive and negative charges. In uniform, static crossed fields, this gives sideways guiding-centre motion; perpendicular $\mathbf{E}$ does not imply unbounded energy gain. In a nonuniform magnetic field, two additional leading drifts are
$$
\mathbf{v}_{\nabla B}=\frac{\mu}{QB}\,\mathbf{b}\times\nabla B,
\qquad
\mathbf{v}_{\rm curv}=\frac{m v_\parallel^2}{QB}\,
\mathbf{b}\times\boldsymbol{\kappa},\qquad
\boldsymbol{\kappa}=(\mathbf{b}\cdot\nabla)\mathbf{b}.
$$
These depend on charge sign. The formulas assume $\rho_L/L_B\ll1$, with $L_B$ a characteristic magnetic-field variation length. They diagnose the full Lorentz orbit; do not add them as extra forces to the existing full-orbit solver, which already captures magnetic drifts.

### 5. Useful electric-field controls

The implemented field is
$$
\mathbf E(\mathbf x,t)=E_{\phi0}\frac{R_0}{R}\cos(\omega t+\varphi)\hat{\boldsymbol\phi}
-k_E[(R-R_0)\hat{\mathbf R}+Z\hat{\mathbf Z}].
$$
`omega=0` and `phase=0` give a steady toroidal drive. Its components are:

- **Toroidal drive:** $\mathbf{E}=E_{\phi0}(t)(R_0/R)\,\hat{\boldsymbol\phi}$, where $\hat{\boldsymbol\phi}=(-Y/R,X/R,0)$. Its component along the predominantly toroidal magnetic field changes parallel energy. The loop voltage is $V_{\rm loop}=\oint\mathbf{E}\cdot d\boldsymbol\ell=2\pi R_0E_{\phi0}$. This represents an externally driven inductive field, not a globally single-valued electrostatic potential around the torus.
- **Radial potential:** choose $\Phi(r)=\tfrac12 k_Er^2$, with $r=\sqrt{(R-R_0)^2+Z^2}$. Then $\mathbf{E}=-\nabla\Phi=-k_E[(R-R_0)\hat{\mathbf R}+Z\hat{\mathbf Z}]$. This smooth field produces an $\mathbf E\times\mathbf B$ drift without a singular direction at $r=0$.

A time-dependent drive can be prescribed as a pulse or sinusoid. Resonant heating requires suitable frequency, polarization, and particle coupling; simply making $\mathbf E$ oscillate does not guarantee heating. If $\mathbf B$ is also varied in time, the electric and magnetic fields must satisfy $\nabla\times\mathbf E=-\partial\mathbf B/\partial t$.

### 6. Including electric acceleration in the Boris update

Keep the drift–kick–drift structure. First set $\mathbf{x}_{1/2}=\mathbf{x}_n+\tfrac12\Delta t\mathbf{v}_n$ and evaluate both fields there at $t_n+\Delta t/2$. Then use
$$
\begin{aligned}
\mathbf{v}^- &=\mathbf{v}_n+\frac{Q\Delta t}{2m}\mathbf{E}_{1/2},\\
\boldsymbol\tau&=\frac{Q\Delta t}{2m}\mathbf{B}_{1/2},\qquad
\mathbf{s}=\frac{2\boldsymbol\tau}{1+|\boldsymbol\tau|^2},\\
\mathbf{v}'&=\mathbf{v}^-+\mathbf{v}^-\times\boldsymbol\tau,\\
\mathbf{v}^+&=\mathbf{v}^-+\mathbf{v}'\times\mathbf{s},\\
\mathbf{v}_{n+1}&=\mathbf{v}^++\frac{Q\Delta t}{2m}\mathbf{E}_{1/2},\\
\mathbf{x}_{n+1}&=\mathbf{x}_{1/2}+\tfrac12\Delta t\mathbf{v}_{n+1}.
\end{aligned}
$$
The two electric half-kicks surround the magnetic rotation. Setting $\mathbf E=0$ recovers the existing update.

### 7. Controls and measurements

| Control | Main quantity to examine |
|---|---|
| `B_0` | Gyrofrequency, Larmor radius, magnetic drift and wall loss |
| `q_s` | Field-line twist; smaller `q_s` gives a stronger poloidal component in this model |
| `v_parallel`, `v_perpendicular` | Initial energy, pitch angle, passing or trapped behaviour |
| Toroidal drive amplitude and pulse duration | Parallel acceleration and energy gain |
| Radial potential strength `k_E` | Cross-field drift and displacement of the orbit |
| `dt`, `t_max` | Time-step convergence and duration of observed confinement |

Store $K(t)$, $v_\parallel(t)$, $v_{\perp,\rm gyro}(t)$, $\rho_L(t)$, $\mu(t)$, and the estimated guiding centre. For wall proximity, track
$$
\frac{r(t)}{a}=\frac{\sqrt{(R(t)-R_0)^2+Z(t)^2}}{a}.
$$
With $\mathbf E=0$, check constant energy; with $\mathbf E\ne0$, compare $K(t)-K(0)$ with $\int_0^t Q\mathbf E\cdot\mathbf v\,dt$. Resolve the fastest local gyration ($\Omega_c\Delta t\ll1$), field variation, and imposed drive; halve `dt` to check orbit convergence. Electric acceleration can increase orbit width and lead to a wall hit. Strong acceleration eventually requires a relativistic momentum-based model.

References: Richard Fitzpatrick's [uniform-field motion](https://farside.ph.utexas.edu/teaching/plasma/Plasma/node15.html), [guiding-centre motion](https://farside.ph.utexas.edu/teaching/plasma/Plasma/node17.html), and [magnetic drifts](https://farside.ph.utexas.edu/teaching/plasma/Plasma/node18.html).


In [ ]:
# Geometry and field (normalized units).
R_0, a = 3.0, 1.0
B_0, q_s = 15.0, 2.0
Q, m = 1.0, 1.0

# Electric-field controls (normalized units). Zero amplitudes give E = 0.
E_params = {
    "E_phi0": 0.02,  # Toroidal drive amplitude; try 0.02 for acceleration.
    "k_E": 0.0,     # Radial potential strength: Phi = k_E * r**2 / 2.
    "omega": 0.0,   # Toroidal drive angular frequency; 0 gives a steady drive.
    "phase": 0.0,   # Toroidal drive phase in radians.
}


In [ ]:

def tokamak_field(position, R_0=R_0, B_0=B_0, q_s=q_s):
    """Cartesian magnetic field of an axisymmetric circular tokamak model."""
    X, Y, Z = np.asarray(position, dtype=float)
    R = np.hypot(X, Y)
    if R <= 1e-12:
        raise ValueError("The field is singular at R=0, outside the vessel.")
    B_R = -B_0 * Z / (q_s * R)
    B_phi = B_0 * R_0 / R
    B_Z = B_0 * (R - R_0) / (q_s * R)
    return np.array([B_R * X / R - B_phi * Y / R,
                     B_R * Y / R + B_phi * X / R, B_Z])


def electric_field(position, time, *, E_phi0=0.0, k_E=0.0,
                   omega=0.0, phase=0.0):
    """Toroidal drive plus a smooth electrostatic minor-radial field."""
    X, Y, Z = np.asarray(position, dtype=float)
    R = np.hypot(X, Y)
    if R <= 1e-12:
        raise ValueError("The electric-field model requires R > 0.")
    e_R = np.array([X / R, Y / R, 0.0])
    e_phi = np.array([-Y / R, X / R, 0.0])
    E_phi = E_phi0 * R_0 / R * np.cos(omega * time + phase)
    return E_phi * e_phi - k_E * ((R - R_0) * e_R + np.array([0.0, 0.0, Z]))


def minor_radius(position):
    return np.hypot(np.hypot(position[0], position[1]) - R_0, position[2])


def tokamak_motion(x_0, v_0, *, Q=Q, m=m, dt=0.005, t_max=60,
                   E_params=None):
    """Boris update with electric half-kicks; stop at the vessel wall."""
    if E_params is None:
        E_params = globals()["E_params"].copy()
    if m <= 0 or dt <= 0 or t_max <= 0:
        raise ValueError("m, dt and t_max must be positive.")
    if minor_radius(x_0) >= a:
        raise ValueError("Start the particle inside the vessel.")
    t = np.linspace(0, t_max, int(np.ceil(t_max / dt)) + 1)
    x = np.zeros((len(t), 3))
    v = np.zeros_like(x)
    x[0], v[0] = x_0, v_0
    hit_wall = False
    for i, step in enumerate(np.diff(t)):
        midpoint = x[i] + 0.5 * step * v[i]
        E_half = electric_field(midpoint, t[i] + step / 2, **E_params)
        electric_kick = Q * E_half * step / (2 * m)
        v_minus = v[i] + electric_kick
        rotation = Q * tokamak_field(midpoint) * step / (2 * m)
        s = 2 * rotation / (1 + np.dot(rotation, rotation))
        v_prime = v_minus + np.cross(v_minus, rotation)
        v_plus = v_minus + np.cross(v_prime, s)
        v[i + 1] = v_plus + electric_kick
        x[i + 1] = midpoint + 0.5 * step * v[i + 1]
        if minor_radius(x[i + 1]) >= a:
            x, v, t = x[:i + 2], v[:i + 2], t[:i + 2]
            hit_wall = True
            break
    return x, v, t, hit_wall


# A passing particle: mostly parallel motion plus a small gyro velocity.
x_0 = np.array([R_0 + 0.45 * a, 0.0, 0.0])
b_hat = tokamak_field(x_0)
b_hat /= np.linalg.norm(b_hat)
v_parallel, v_perpendicular = 1.0, 0.35
v_0 = v_parallel * b_hat + np.array([v_perpendicular, 0.0, 0.0])
x, v, t, hit_wall = tokamak_motion(x_0, v_0, t_max=80)
R = np.hypot(x[:, 0], x[:, 1])
r = np.hypot(R - R_0, x[:, 2])
print(f"Simulated to t={t[-1]:.2f}; max r/a={r.max() / a:.3f}; wall hit: {hit_wall}")


In [ ]:
# Electric and magnetic field lines, before the particle animation.
field_time = 1.0  # Electric-field snapshot time; change for an oscillating drive.


def trace_field_line(field, seed, *, ds=0.04 * a, length=4 * np.pi * R_0):
    """Trace both directions with RK4: dx/ds = F / |F| inside the vessel."""
    def direction(point):
        vector = np.asarray(field(point), dtype=float)
        strength = np.linalg.norm(vector)
        return vector / strength if strength > 1e-12 else np.zeros(3)

    branches = []
    for sign in (-1, 1):
        point = np.asarray(seed, dtype=float).copy()
        points = [point.copy()]
        step = sign * ds
        for _ in range(int(np.ceil(length / (2 * ds)))):
            k1 = direction(point)
            if np.linalg.norm(k1) == 0:
                break
            k2 = direction(point + step * k1 / 2)
            k3 = direction(point + step * k2 / 2)
            k4 = direction(point + step * k3)
            next_point = point + step * (k1 + 2 * k2 + 2 * k3 + k4) / 6
            if minor_radius(next_point) >= a:
                break
            if np.linalg.norm(next_point - point) < 1e-12:
                break
            points.append(next_point.copy())
            point = next_point
        branches.append(np.asarray(points))
    return np.vstack([branches[0][:0:-1], branches[1]])


# Seeds on two nested magnetic surfaces, at opposite poloidal angles.
field_seeds = [np.array([R_0 + radius * np.cos(theta), 0,
                         radius * np.sin(theta)])
               for radius in (0.3 * a, 0.65 * a) for theta in (0, np.pi)]
field_snapshot = lambda point: electric_field(point, field_time, **E_params)

with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 9,
                     "grid.color": "#28405a", "grid.linewidth": 0.5}):
    fields_fig = plt.figure(figsize=(12, 5.5), facecolor="#080e1c")
    fields_fig.suptitle(f"Tokamak field lines  /  t = {field_time:g}",
                        color="#edf5ff", fontsize=20, weight="bold", y=0.96)
    phi_grid, theta_grid = np.meshgrid(np.linspace(0, 2 * np.pi, 49),
                                       np.linspace(0, 2 * np.pi, 17))
    vessel_R = R_0 + a * np.cos(theta_grid)
    field_lines = {}
    for panel, (label, field, color) in enumerate([
            ("MAGNETIC FIELD B", tokamak_field, "#6ce6ef"),
            ("ELECTRIC FIELD E", field_snapshot, "#ffc56e")], start=1):
        field_ax = fields_fig.add_subplot(1, 2, panel, projection="3d",
                                          facecolor="#080e1c")
        field_ax.set_title(label, color=color, fontsize=11, pad=8)
        field_ax.plot_wireframe(vessel_R * np.cos(phi_grid),
                                 vessel_R * np.sin(phi_grid), a * np.sin(theta_grid),
                                 rstride=2, cstride=4, color="#4d8cb1",
                                 alpha=0.18, linewidth=0.5)
        lines = [trace_field_line(field, seed) for seed in field_seeds]
        field_lines[label] = lines
        visible = False
        for line in lines:
            if len(line) < 2:
                continue
            visible = True
            field_ax.plot(*line.T, color=color, lw=1.3, alpha=0.85)
            # Arrowheads indicate the actual field direction.
            for j in np.linspace(0, len(line) - 1, 3, dtype=int):
                vector = field(line[j])
                norm = np.linalg.norm(vector)
                if norm > 1e-12:
                    field_ax.quiver(*line[j], *(vector / norm), length=0.25 * a,
                                    color=color, linewidth=1.1, arrow_length_ratio=0.5)
        if not visible:
            field_ax.text2D(0.17, 0.5, "Field is zero at the snapshot time",
                             transform=field_ax.transAxes, color=color)
        limit = R_0 + 1.12 * a
        field_ax.set(xlim=(-limit, limit), ylim=(-limit, limit),
                     zlim=(-1.3 * a, 1.3 * a))
        field_ax.set_box_aspect((2 * limit, 2 * limit, 2.6 * a))
        field_ax.set_proj_type("ortho")
        field_ax.view_init(elev=29, azim=-48)
        for axis, name in zip((field_ax.xaxis, field_ax.yaxis, field_ax.zaxis),
                               ("x", "y", "z")):
            axis.set_pane_color((0.03, 0.055, 0.10, 1))
            axis.set_label_text(name)
            axis.label.set_color("#879bb5")
            axis.line.set_color("#28405a")
            axis.set_major_locator(MaxNLocator(3))
        field_ax.tick_params(colors="#879bb5", labelsize=8)
    fields_fig.text(0.08, 0.05, "Arrows show direction; line spacing does not encode field strength.",
                     color="#879bb5", fontsize=9)
    fields_fig.subplots_adjust(left=0.02, right=0.98, bottom=0.12, top=0.85, wspace=0.02)
    plt.show()


In [ ]:
# Three-dimensional vessel and an R-Z cross-section, with a time slider.
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "grid.color": "#28405a", "grid.alpha": 0.35,
                     "grid.linewidth": 0.5, "animation.embed_limit": 100}):
    fig = plt.figure(figsize=(10, 6.5), dpi=85, facecolor="#080e1c")
    ax = fig.add_axes([0.01, 0.12, 0.71, 0.68], projection="3d",
                      facecolor="#080e1c")
    section = fig.add_axes([0.76, 0.32, 0.21, 0.35], facecolor="#0c1629")
    fig.text(0.06, 0.94, "TOKAMAK / SINGLE PARTICLE", fontsize=10,
             color="#6ce6ef", weight="bold")
    fig.text(0.06, 0.875, "An orbit inside a magnetic torus", fontsize=23,
             color="#edf5ff", weight="bold")
    fig.text(0.06, 0.045, "BORIS INTEGRATION   /   PRESCRIBED FIELD   /   NORMALIZED UNITS",
             fontsize=9, color="#879bb5")

    phi, theta = np.meshgrid(np.linspace(0, 2 * np.pi, 65),
                              np.linspace(0, 2 * np.pi, 19))
    shell_R = R_0 + a * np.cos(theta)
    ax.plot_wireframe(shell_R * np.cos(phi), shell_R * np.sin(phi),
                      a * np.sin(theta), rstride=2, cstride=4,
                      color="#4d8cb1", alpha=0.22, linewidth=0.6)
    angle = np.linspace(0, 2 * np.pi, 300)
    ax.plot(R_0 * np.cos(angle), R_0 * np.sin(angle), np.zeros_like(angle),
             color="#6ce6ef", alpha=0.25, lw=1, linestyle="--")
    ax.plot(*x.T, color="#ffc56e", alpha=0.15, lw=0.6)
    limit = R_0 + 1.12 * a
    ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-1.3 * a, 1.3 * a))
    ax.set_box_aspect((2 * limit, 2 * limit, 2.6 * a))
    ax.set_proj_type("ortho")
    ax.view_init(elev=29, azim=-48)
    for axis, label in zip((ax.xaxis, ax.yaxis, ax.zaxis), ("x", "y", "z")):
        axis.set_pane_color((0.03, 0.055, 0.10, 1))
        axis.set_label_text(label)
        axis.label.set_color("#879bb5")
        axis.line.set_color("#28405a")
        axis.set_major_locator(MaxNLocator(3))
    ax.tick_params(colors="#879bb5", labelsize=8)

    section.plot(R_0 + a * np.cos(angle), a * np.sin(angle),
                  color="#6ce6ef", lw=1.5, alpha=0.65)
    section.plot(R_0, 0, "+", color="#879bb5")
    section.plot(R, x[:, 2], color="#ffc56e", alpha=0.15, lw=0.7)
    section.set(xlim=(R_0 - 1.15 * a, R_0 + 1.15 * a),
                 ylim=(-1.15 * a, 1.15 * a), xlabel="R", ylabel="Z")
    section.set_aspect("equal")
    section.set_title("POLOIDAL VIEW", color="#6ce6ef", fontsize=10, pad=13)
    section.tick_params(colors="#879bb5", labelsize=8)
    section.xaxis.label.set_color("#879bb5")
    section.yaxis.label.set_color("#879bb5")
    section.xaxis.set_major_locator(MaxNLocator(3))
    section.yaxis.set_major_locator(MaxNLocator(3))
    for spine in section.spines.values():
        spine.set_color("#28405a")
    section.grid(alpha=0.25)

    trail_glow, = ax.plot([], [], [], color="#ffc56e", lw=7, alpha=0.08)
    trail, = ax.plot([], [], [], color="#ffc56e", lw=1.7, alpha=0.95)
    halo, = ax.plot([], [], [], "o", ms=23, color="#ff85c8", alpha=0.12)
    particle, = ax.plot([], [], [], "o", ms=7, color="#ffd0e9",
                        markeredgecolor="#ff85c8")
    section_trail, = section.plot([], [], color="#ffc56e", lw=1.1)
    section_ball, = section.plot([], [], "o", ms=6, color="#ffd0e9")
    readout = fig.text(0.765, 0.22, "", color="#d3e5f8", fontsize=10,
                        family="monospace", va="top")
    fig.text(0.765, 0.74, "TOROIDAL + POLOIDAL\nMAGNETIC FIELD", color="#879bb5", fontsize=9)
    fps = 24
    # Compress 60 simulation time units into a 12-second movie.
    frames = np.unique(np.linspace(0, len(t) - 1, 12 * fps + 1).astype(int))
    initial_speed = np.linalg.norm(v[0])

    def update(i):
        start = np.searchsorted(t, max(0, t[i] - 16))
        for line in (trail_glow, trail):
            line.set_data_3d(*x[start:i + 1].T)
        for point in (halo, particle):
            point.set_data_3d(*x[i:i + 1].T)
        section_trail.set_data(R[:i + 1], x[:i + 1, 2])
        section_ball.set_data([R[i]], [x[i, 2]])
        status = "WALL HIT" if hit_wall and i == len(t) - 1 else "IN VESSEL"
        readout.set_text(f"t     {t[i]:6.2f}\nr/a   {r[i] / a:6.3f}\n"
                         f"|v|   {np.linalg.norm(v[i]):6.3f}\n{status}")
        return trail_glow, trail, halo, particle, section_trail, section_ball, readout

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps,
                               blit=False, repeat=True)
    player = animation.to_jshtml(default_mode="loop")
    plt.close(fig)

display(HTML(player))


In [ ]:
# Running this cell saves both the standalone HTML player and MP4.
import imageio_ffmpeg
from tokamak_player import make_tokamak_player

controlled_player = make_tokamak_player(R_0, a, B_0, q_s, Q, m, E_params, x, v, t)
Path("tokamak.html").write_text(controlled_player, encoding="utf-8")
with plt.rc_context({"animation.ffmpeg_path": imageio_ffmpeg.get_ffmpeg_exe()}):
    animation.save("tokamak.mp4", writer=FFMpegWriter(fps=fps), dpi=120)

display(Video("tokamak.mp4", embed=True))
